# SoyCare AI: Soybean Leaf Disease Detection
## End-to-End Model Training, Evaluation & Grad-CAM Explainability Pipeline

**Author:** Aman Yadav  
**Project:** SoyCare AI Prototype  
**Architecture:** EfficientNetB0 Transfer Learning & Fine-Tuning  
**Task:** 6-Class Multi-class Foliar Disease Classification on Soybean (*Glycine max*)

---
### Pipeline Overview:
1. **Environment Setup & Hyperparameters**
2. **Data Ingestion & Augmentation**
3. **Model Construction (EfficientNetB0 Backbone)**
4. **Stage 1: Feature Extraction**
5. **Stage 2: Deep Layer Fine-Tuning**
6. **Model Evaluation & Confusion Matrix**
7. **Grad-CAM Visual Explainability**

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
print(f"Available GPUs: {tf.config.list_physical_devices('GPU')}")

# Constants & Hyperparameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE
DATA_DIR = Path("../data/processed")
CLASSES = ['Bacterial blight', 'Downy mildew', 'Frogeye leaf spot', 'Healthy', 'Septoria brown spot', 'Soybean rust']
NUM_CLASSES = len(CLASSES)

## 2. Data Loading & Augmentation Pipeline

In [ ]:
def load_dataset_splits(data_dir):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir / "train",
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=True
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir / "validation",
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )
    test_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir / "test",
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=False
    )
    return train_ds.cache().prefetch(AUTOTUNE), val_ds.cache().prefetch(AUTOTUNE), test_ds.cache().prefetch(AUTOTUNE)

# Data Augmentation Layer to reduce overfitting on field variations
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15)
], name="data_augmentation")

## 3. Model Architecture (EfficientNetB0 Backbone)

In [ ]:
def build_model(num_classes):
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = data_augmentation(inputs)
    
    # Pre-trained EfficientNetB0 without top classifier
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, 
        weights="imagenet", 
        input_tensor=x
    )
    base_model.trainable = False  # Freeze for Stage 1
    
    x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    
    model = tf.keras.Model(inputs, outputs, name="SoyCare_Model")
    return model, base_model

model, base_model = build_model(NUM_CLASSES)
model.summary()

## 4. Stage 1: Feature Extraction Training

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("../models/soybean_disease_model.keras", monitor="val_accuracy", save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
]

# history_stage1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

## 5. Stage 2: Fine-Tuning Top Convolutional Blocks

In [ ]:
# Unfreeze top 30 layers for fine-tuning with low learning rate
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")]
)

# history_stage2 = model.fit(train_ds, validation_data=val_ds, epochs=25, initial_epoch=15, callbacks=callbacks)

## 6. Confusion Matrix & Classification Report

In [ ]:
# Render Confusion Matrix
from src.evaluate import generate_benchmark_metrics
report = generate_benchmark_metrics()
print("Macro F1-Score:", report["macro_avg_f1_score"])
print("Overall Accuracy:", report["overall_accuracy"])

## 7. Grad-CAM Attention Heatmap Visualization

In [ ]:
from src.gradcam import make_gradcam_heatmap, save_and_overlay_gradcam
print("Grad-CAM explainability pipeline loaded successfully.")